# 01 — Baseline and data validation (WS1, due Oct 17)

**Gate:** clean GaitBase Rank-1 on CASIA-B within ±1.0 pp of OpenGait's reported NM/BG/CL
(97.6 / 94.0 / 77.4, `configs/baseline.yaml`), computed through `evaluate_attack(attack=None)`
AND identical to OpenGait's own `evaluate_indoor_dataset` on the same embeddings.
GaitSet: no released CASIA-B checkpoint; it is trained in `01b_gaitset_training` and its gate
row is filled when that finishes (needed before the He et al. validation, Oct 28).

Runtime: Kaggle GPU (T4/P100). All logic lives in `src/`.

In [ ]:
# --- Paths per platform. Edit ONLY this cell. -------------------------------------------
import os, sys, subprocess
from pathlib import Path

PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else \
           "colab" if "google.colab" in sys.modules else "local"
PATHS = {
    # Kaggle: team repo cloned into /kaggle/working; data + checkpoints from the private team Dataset.
    "kaggle": dict(repo="/kaggle/working/cross_attack_gait",
                   opengait="/kaggle/working/OpenGait",
                   dataset_root="/kaggle/input/casia-b-opengait/CASIA-B-pkl",
                   ckpt_dir="/kaggle/input/casia-b-opengait/checkpoints"),
    # Colab: Drive mirror of the same Dataset.
    "colab": dict(repo="/content/cross_attack_gait",
                  opengait="/content/OpenGait",
                  dataset_root="/content/drive/MyDrive/gait/CASIA-B-pkl",
                  ckpt_dir="/content/drive/MyDrive/gait/checkpoints"),
    # Local: set CASIAB_PKL and GAIT_CKPT_DIR (e.g. ../data/CASIA-B-pkl). CPU-only machines cannot run
    # OpenGait models (BaseModel hardcodes CUDA); official gate runs on GPU.
    "local": dict(repo=str(Path.cwd().parent), opengait=str(Path.cwd().parent.parent / "OpenGait"),
                  dataset_root=os.environ.get("CASIAB_PKL", ""), ckpt_dir=os.environ.get("GAIT_CKPT_DIR", "")),
}[PLATFORM]
TEAM_REPO_URL = "https://github.com/haragam22/gait-paper.git"   # e.g. https://github.com/<org>/cross_attack_gait.git (private; use a token on Kaggle)

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
if PLATFORM != "local":
    env = dict(os.environ)
    if PLATFORM == "kaggle":  # private repo: add a GitHub token as Kaggle secret GITHUB_TOKEN (Add-ons > Secrets)
        import base64
        from kaggle_secrets import UserSecretsClient
        auth = base64.b64encode(f"x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}".encode()).decode()
        # token via env-only git config: never in argv, error messages, or .git/config (saved with notebook output)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    if Path(PATHS["repo"]).exists():   # session already has the repo: get the latest code, configs, results
        subprocess.run(["git", "-C", PATHS["repo"], "pull", "--ff-only"], check=True, env=env)
    else:
        subprocess.run(["git", "clone", TEAM_REPO_URL, PATHS["repo"]], check=True, env=env)
sys.path.insert(0, PATHS["repo"]); os.chdir(PATHS["repo"])

import src.utils as U
BASE = U.load_yaml("configs/baseline.yaml")
if not Path(PATHS["opengait"]).exists():
    subprocess.run(["git", "clone", "https://github.com/ShiqiYu/OpenGait.git", PATHS["opengait"]], check=True)
subprocess.run(["git", "-C", PATHS["opengait"], "checkout", "-q", BASE["opengait_commit"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "tensorboard"], check=True)

# Kaggle data: use a packed CASIA-B-pkl attached as input (team Dataset or this notebook's saved output);
# otherwise pack the public PNG mirror (trnquanghuyn/casia-b, attach as input) with OpenGait's own script,
# same command as docs/reproducibility.md (~20 min). "Save Version" keeps /kaggle/working/CASIA-B-pkl so
# teammates attach this notebook's output instead of re-packing.
if PLATFORM == "kaggle" and not Path(PATHS["dataset_root"]).exists():
    # Attached inputs mount at /kaggle/input/<slug>/ or deeper (datasets/<owner>/<slug>/, notebooks/...).
    find_input = lambda rel: (p for pat in ("*/", "*/*/", "*/*/*/", "*/*/*/*/") for p in Path("/kaggle/input").glob(pat + rel))
    attached = next(find_input("CASIA-B-pkl"), None)
    packed = Path("/kaggle/working/CASIA-B-pkl")
    if attached is None and not (packed / ".done").exists():
        raw = next(find_input("output/001"), None)
        if raw is None:
            tree = sorted(str(p) for pat in ("*", "*/*", "*/*/*") for p in Path("/kaggle/input").glob(pat) if p.is_dir())
            raise FileNotFoundError("No CASIA-B-pkl and no raw CASIA-B (output/001) under /kaggle/input. Attach "
                                    "Notebook 01's saved output (Add Input > Your Work) or trnquanghuyn/casia-b. "
                                    "Found:\n" + "\n".join(tree[:40]))
        raw = raw.parent
        subprocess.run([sys.executable, "datasets/pretreatment.py", "-i", str(raw), "-o", str(packed),
                        "-d", "GREW", "-n", str(os.cpu_count())], cwd=PATHS["opengait"], check=True)
        (packed / ".done").touch()
    PATHS["dataset_root"] = str(attached or packed)
print("dataset_root:", PATHS["dataset_root"])
BASE["opengait_dir"] = PATHS["opengait"]
BASE["dataset_root"] = PATHS["dataset_root"]

In [ ]:
# --- Cell 1 record: date, seed, versions, GPU, platform, OpenGait commit --------------
SEED = BASE["seed"]
U.seed_everything(SEED)
ENV = U.env_record(SEED, BASE["opengait_dir"], {"notebook": "01_baseline_and_data_validation"})
assert ENV["opengait_commit"] == BASE["opengait_commit"], ENV["opengait_commit"]
ENV

In [ ]:
# --- Checkpoint: from the team Dataset, else download the released one -----------------
import urllib.request
gb = BASE["models"]["gaitbase"]
ck = Path(PATHS["ckpt_dir"]) / "GaitBase_DA-60000.pt"
if not ck.exists():
    ck = Path("/kaggle/working" if PLATFORM == "kaggle" else ".") / "GaitBase_DA-60000.pt"
    if not ck.exists():
        urllib.request.urlretrieve(gb["checkpoint_url"], ck)
gb["checkpoint"] = str(ck)
gs = BASE["models"]["gaitset"]
gs_ck = Path(PATHS["ckpt_dir"]) / "GaitSet-40000.pt"
gs["checkpoint"] = str(gs_ck) if gs_ck.exists() else ""
print("GaitBase:", ck, U.file_sha256(ck)[:12], "| GaitSet:", gs["checkpoint"] or "not trained yet (01b)")

In [ ]:
# --- Load GaitBase (OpenGait code + config + released checkpoint, strict) --------------
from src.evaluation import load_model, Gallery, evaluate_attack, aggregate_rank1, official_opengait_rank1, gate_rows
model, data, og_cfg = load_model(BASE, "gaitbase")
ENV.update(checkpoint=gb["checkpoint"], checkpoint_id=model.checkpoint_id,
           fp16_eval=model.fp16_eval, frames_all_limit=data.frames_all_limit)
print(model.name, model.checkpoint_id, "fp16 eval:", model.fp16_eval, "device:", model.device)

## Data validation

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
S = data.summary()
S["n_frames"] = [data.n_frames(r) for r in data.records]
subjects = sorted(S.subject.unique())
print("test subjects:", len(subjects), subjects[0], "...", subjects[-1])
assert len(subjects) == 50 and subjects[0] == "075" and subjects[-1] == "124"

counts = S.pivot_table(index="subject", columns="seq_type", values="view", aggfunc="count").fillna(0).astype(int)
print("sequences per subject x type (expected 11 views each):")
display(counts[counts.ne(11).any(axis=1)] if counts.ne(11).any(axis=None) else "all complete")
expected = {(s, t, v) for s in subjects for t in counts.columns for v in sorted(S.view.unique())}
missing = sorted(expected - set(zip(S.subject, S.seq_type, S.view)))
print(f"{len(S)} sequences, {len(missing)} missing:", missing[:20])
print("gallery:", (S.role == "gallery").sum(), "probes:", (S.role == "probe").sum(),
      S[S.role == "probe"].condition.value_counts().to_dict())

In [ ]:
x0 = data.load(data.records[0])
print("tensor shape [1,T,H,W]:", tuple(x0.shape), x0.dtype, float(x0.min()), float(x0.max()))
assert x0.shape[2:] == (64, 44)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
S.n_frames.hist(bins=60, ax=ax[0]); ax[0].set_title("frames per sequence"); ax[0].set_xlabel("T")
S.groupby("condition").n_frames.plot.kde(ax=ax[1], legend=True); ax[1].set_title("by condition")
print(S.n_frames.describe().round(1).to_dict(), "| > frames_all_limit:", int((S.n_frames > data.frames_all_limit).sum()))

In [ ]:
# 3 sequences per condition: 8 evenly spaced frames each. Check frame order and broken frames.
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(9, 8, figsize=(10, 14))
row = 0
for cond, types in BASE["probe_types"].items():
    pool = [r for r in data.probes if r.seq_type in types]
    for r in [pool[i] for i in rng.choice(len(pool), 3, replace=False)]:
        x = data.load(r)[0]
        for j, t in enumerate(np.linspace(0, x.shape[0] - 1, 8).astype(int)):
            axes[row, j].imshow(x[t], cmap="gray", vmin=0, vmax=1); axes[row, j].axis("off")
        axes[row, 0].set_title(f"{r.sample_id} T={x.shape[0]}", fontsize=7, loc="left")
        row += 1
plt.tight_layout()

# Broken frames: empty, or silhouette area far from the sequence median.
bad = []
for r in data.records:
    area = data.load(r)[0].flatten(1).mean(1).numpy()
    med = np.median(area)
    n_bad = int(((area < 0.2 * med) | (area > 3 * med)).sum())
    if n_bad:
        bad.append((r.sample_id, n_bad, len(area)))
print(f"sequences with suspicious frames: {len(bad)} / {len(data.records)}", bad[:10])

## Clean evaluation via `evaluate_attack(attack=None)`

In [ ]:
RAW = Path("results/raw")
gallery = Gallery.build(model, data, cache=f"results/processed/gallery_{model.name}_{model.checkpoint_id}.npz")
clean = evaluate_attack(model, data, gallery, data.probes, attack=None, attack_name="clean", tier="-",
                        seed=SEED, out_csv=RAW / "baseline_results.csv",
                        run_id=f"clean-{model.name}-{model.checkpoint_id}", run_meta={"env": ENV})
table = aggregate_rank1(clean)
display(table.round(2))

In [ ]:
# Cross-check against OpenGait's own evaluator on the same embeddings. `ours` (fp32 distances) must equal
# OpenGait's evaluator at fp32 distance exactly; OpenGait's stock fp16-distance numbers are shown alongside
# (differ by near-tie rounding only, see docs/deviations.md).
official = official_opengait_rank1(model, data)
gate = pd.DataFrame(gate_rows("gaitbase", table, official, gb["reported_rank1"], BASE["gate_tolerance_pct"]))
display(gate)
print("GATE (GaitBase):", "PASS" if (gate.matches_evaluator & gate.within_tolerance).all() else "FAIL")

In [ ]:
# GaitSet row — runs only once 01b has produced GaitSet-40000.pt in the team Dataset.
if gs["checkpoint"]:
    gs_model, gs_data, _ = load_model(BASE, "gaitset")
    gs_gallery = Gallery.build(gs_model, gs_data, cache=f"results/processed/gallery_gaitset_{gs_model.checkpoint_id}.npz")
    gs_clean = evaluate_attack(gs_model, gs_data, gs_gallery, gs_data.probes, attack=None, attack_name="clean",
                               seed=SEED, out_csv=RAW / "baseline_results.csv",
                               run_id=f"clean-gaitset-{gs_model.checkpoint_id}", run_meta={"env": ENV})
    gs_table, gs_off = aggregate_rank1(gs_clean), official_opengait_rank1(gs_model, gs_data)
    gate = pd.concat([gate, pd.DataFrame(gate_rows("gaitset", gs_table, gs_off, gs["reported_rank1"],
                                                   BASE["gate_tolerance_pct"]))], ignore_index=True)
    display(gate)
else:
    print("GaitSet skipped: train it in 01b_gaitset_training first.")

In [ ]:
U.save_json({**ENV, "gate": gate.to_dict("records"), "per_view_gaitbase": table.round(2).to_dict(),
             "n_sequences": len(S), "missing_sequences": missing, "suspicious_frames": bad},
            RAW / "env_baseline.json")
print("saved", RAW / "baseline_results.csv", RAW / "env_baseline.json")

In [ ]:
# Push results/ to GitHub so they can be pulled locally (`git pull`). Needs secret GITHUB_TOKEN with
# Contents: read AND write on the repo. Large artefacts (*.npz, *.pt) are gitignored and stay on Kaggle.
pushed = U.push_results(PATHS["repo"], f"results: Notebook 01 clean baseline ({ENV['platform']}, {ENV['date_utc']})",
                        U.github_token(PLATFORM))
print("pushed results/ to GitHub" if pushed else "no new results to push")

## Findings, limitations, next step
Numbers from `results/raw/env_baseline.json` (Kaggle T4, 2026-10-16, repo commit 1c8d6a3).

- **Gate: PASS (GaitBase).** `evaluate_attack(attack=None)` Rank-1 NM / BG / CL = 97.87 / 93.85 / 77.03; identical to
  OpenGait's `evaluate_indoor_dataset` at fp32 distance; OpenGait's stock fp16-distance evaluator gives 97.90 / 93.87 / 77.01
  (near-tie rounding, recorded in `docs/deviations.md`). Within 1.0 pp of reported 97.6 / 94.0 / 77.4 (diff +0.27 / -0.15 / -0.37).
- Data: 50 test subjects; 15 test sequences missing (079 bg-02 ×2, 088 nm-03/04 ×9, 109 nm-01 ×4) — same list as
  `docs/reproducibility.md`; 3,298 probes evaluated. 1 suspicious-frame flag (096-nm-04-144, 1 of 71 frames).
- Limitations: GaitSet row pending (Notebook 01b).
- Next: Notebook 02 (PGD), due Oct 20; GaitSet row when `GaitSet-40000.pt` exists.